#  Petit agent avec outils (smolagents)

**Objectif :** construire un agent débutant qui **appelle des outils** pour répondre,
avec le framework **smolagents** — 100 % open source, aucune clé API, petit modèle local.



## 0) Installation

`smolagents[transformers]` installe smolagents **avec** transformers/torch
(indispensable pour charger un modèle local). `wikipedia` est inclus par l'énoncé
(non utilisé ici, mais inoffensif).

In [1]:
# À lancer une seule fois par session Colab
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 9.7 MB/s eta 0:00:00


## 1) Définir la base de connaissances (KB)

La KB est une simple **liste de dictionnaires**. Chaque entrée a :
- `source` : un identifiant court (ex. `kb:agentic`) → sert à **citer** la source ;
- `text`   : le contenu de l'extrait.

> Vous pouvez ajouter vos propres extraits (5 à 8 au total).

In [2]:
# Petite base de connaissances (5 extraits). N'hésitez pas à en ajouter.
kb_snippets = [
    {'source': 'kb:agentic',  'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools',    'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity',  'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]

print('Nombre d\'entrées dans la KB :', len(kb_snippets))

Nombre d'entrées dans la KB : 5


## 2) Définir les deux outils

Dans smolagents, un outil est une **sous-classe de `Tool`**. Elle DOIT définir :
- `name` : le nom que l'agent utilisera pour appeler l'outil ;
- `description` : à quoi sert l'outil (l'agent la lit pour décider) ;
- `inputs` : le **schéma des arguments** (type + description de chacun) ;
- `output_type` : le type de sortie (ici `"string"`) ;
- une méthode `forward(...)` : le code réellement exécuté.

>  **Pièges corrigés ici :**
> 1. `inputs` et `output_type` sont **obligatoires** (leur absence provoque
>    `TypeError: You must set an attribute inputs`).
> 2. Tout argument ayant une **valeur par défaut** (ici `op="add"`) doit avoir
>    `"nullable": True` dans son schéma, sinon smolagents lève une `AssertionError`.

In [3]:
from smolagents import Tool


class KBLookupTool(Tool):
    """Outil 1 : cherche dans la base de connaissances les extraits contenant les mots de la requête."""
    name = "kb_lookup_tool"
    description = "Looks up relevant snippets from the custom knowledge base. Returns them with a [source] tag."
    # Schéma de l'unique argument attendu :
    inputs = {
        "query": {"type": "string", "description": "Keywords to search for in the knowledge base"},
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()          # IMPORTANT : appelle la validation interne de smolagents
        self.kb = kb                # on garde la KB à l'intérieur de l'outil

    def forward(self, query: str) -> str:
        # On découpe la requête en mots, en ignorant les mots trop courts et la ponctuation.
        mots = [w.strip("?.,!;:").lower() for w in query.lower().split()]
        mots = [w for w in mots if len(w) >= 3]   # on retire "is", "an", "ai"... (bruit)

        # On garde les extraits dont le texte contient au moins un de ces mots.
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in mots)
        ]
        # On renvoie les extraits (avec leur tag de source), ou un message clair si rien trouvé.
        return "\n".join(matches) if matches else "No KB match."


class MathTool(Tool):
    """Outil 2 : additionne (add) ou multiplie (multiply) deux nombres."""
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a":  {"type": "number", "description": "First number"},
        "b":  {"type": "number", "description": "Second number"},
        # 'op' a une valeur par défaut -> il FAUT "nullable": True
        "op": {"type": "string", "description": "Operation: add or multiply",
               "enum": ["add", "multiply"], "nullable": True},
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


# On instancie les deux outils.
kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()
print(" Outils prêts :", kb_tool.name, "et", math_tool.name)

 Outils prêts : kb_lookup_tool et math_tool


## 3) Charger un petit modèle local

L'agent a besoin d'un « cerveau » (un LLM) pour décider quel outil appeler.
On le charge en local avec `TransformersModel` (aucune clé API).

>  **À propos du modèle :** l'énoncé cite `sshleifer/tiny-gpt2`. C'est un modèle
> **de test, non entraîné** : il produit du charabia et **ne sait pas** générer des
> appels d'outils → l'agent ne fonctionnerait pas. On choisit donc par défaut un
> petit modèle **« instruct »** capable de suivre des instructions et d'appeler des
> outils, tout en restant léger et local.
>
>  Les petits modèles restent limités : les appels d'outils ne réussissent pas
> toujours. Pour un résultat plus fiable, augmentez la taille du modèle.

In [4]:
from smolagents import TransformersModel

# --- Choix du modèle (changez cette ligne pour tester d'autres modèles) ---
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # petit modèle instruct (~1 Go), fonctionne pour la démo
# Alternatives :
#   MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # un peu plus gros (~2 Go), plus bavard
#   MODEL_ID = "sshleifer/tiny-gpt2"                   # ultra léger MAIS charabia (ne pilote pas les outils)

# device_map="auto" : utilise le GPU (T4) s'il est disponible, sinon le CPU.
model = TransformersModel(
    model_id=MODEL_ID,
    device_map="auto",
    max_new_tokens=1024,      # limite la longueur des réponses (plus rapide)
)

print(" Modèle prêt :", MODEL_ID)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

 Modèle prêt : Qwen/Qwen2.5-0.5B-Instruct


## 4) Créer l'agent

`ToolCallingAgent` reçoit la liste des outils et le modèle. Les `instructions`
lui rappellent les règles de l'énoncé : **citer la KB**, **rester concis**, et
**signaler le manque de preuves** en proposant une question de suivi.

- `max_steps=3` : au plus 3 étapes de raisonnement (évite les boucles infinies).

In [5]:
from smolagents import ToolCallingAgent

agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],   # les outils que l'agent peut appeler
    model=model,                  # le "cerveau" chargé à l'étape 3
    max_steps=3,                  # nombre maximum d'étapes
    instructions=(
        "You are an agentic AI that uses tools to answer questions. "
        "When you use the knowledge base, always cite the source tag, e.g. [kb:agentic]. "
        "Keep answers concise (2-4 sentences). "
        "If evidence is missing, say so and propose a follow-up question."
    ),
)

print(" Agent prêt avec les outils :", [t.name for t in [kb_tool, math_tool]])

 Agent prêt avec les outils : ['kb_lookup_tool', 'math_tool']


## 5) Questions de test + inspection des appels d'outils

On pose 3 questions :
1. une **addition** → l'agent devrait appeler `math_tool` ;
2. une **multiplication** → `math_tool` ;
3. une question de **connaissance** → `kb_lookup_tool`, avec citation `[kb:...]`.

Après chaque réponse, on lit `agent.memory.steps` pour **afficher quels outils
ont réellement été appelés** (nom + arguments).

>  smolagents affiche aussi ses étapes en direct dans la sortie de la cellule.

In [6]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("=" * 70)
    print("Q :", q)
    try:
        # On (ré)initialise la mémoire pour n'inspecter que les étapes de CETTE question.
        result = agent.run(q, reset=True)

        # On parcourt la mémoire pour lister les appels d'outils réellement effectués.
        appels = []
        for step in agent.memory.steps:
            for tc in (getattr(step, "tool_calls", None) or []):
                appels.append(f"{tc.name}({tc.arguments})")

        print("Outils appelés :", appels if appels else "(aucun)")
        print("Réponse        :", result)
    except Exception as e:
        # On ne plante pas la boucle si une question échoue (petit modèle = parfois instable).
        print(" Erreur pour cette question :", type(e).__name__, "-", e)
print("=" * 70)

Q : Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-0.5B-Instruct ────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': 12, 'b': 30}                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'An agentic AI loop is a process where an artificial    │
│ intelligence system acts without being explicitly programmed to do so. In other words, an AI loop involves      │
│ executing a set of instructions repeatedly until a specific condition is met.'}                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: An agentic AI loop is a process where an artificial intelligence system acts without being explicitly
programmed to do so. In other words, an AI loop involves executing a set of instructions repeatedly until a 
specific condition is met.

Final answer: An agentic AI loop is a process where an artificial intelligence system acts without being explicitly
programmed to do so. In other words, an AI loop involves executing a set of instructions repeatedly until a 
specific condition is met.

[Step 2: Duration 2.81 seconds| Input tokens: 2,898 | Output tokens: 94]

Outils appelés : ["math_tool({'a': 1, 'b': 2, 'op': 'multiply'})", "final_answer({'answer': 'An agentic AI loop is a process where an artificial intelligence system acts without being explicitly programmed to do so. In other words, an AI loop involves executing a set of instructions repeatedly until a specific condition is met.'})"]
Réponse        : An agentic AI loop is a process where an artificial intelligence system acts without being explicitly programmed to do so. In other words, an AI loop involves executing a set of instructions repeatedly until a specific condition is met.
